# Test 2 of the Occlusion Analysis in ResNet18 (Unencrypted Version)

This Jupyter Notebook consists of the tests for the three algorithms for the unencrypted Occlusion Analysis on ResNet18.

ResNet18 is a neural network that was trained on the ImageNet dataset.

The three algorithms used are the sliding-window algorithm, the hierarchical algorithm, and the Monte Carlo algorithm. They use two parties for the inference, but none of them encrypt their data.

The cells of the presets must be run before the algorithms of the occlusion analysis. The algorithms don't need to be executed in a specific order.

Please note that this code was used to test the metrics of sparsity, robustness and consistency. The description of the algorithms can be seen in the non-test Jupyter notebooks.

## Presets

The presets contain all the important imports and a function to process the picture so that they are suitable for the inference.

The function process_image takes the picture from a specific path and transforms it so it can be used in the inference with ResNet18. It saves the picture to a specific save path.

In [ ]:
# PyTorch and Torchvision
import torch
import torchvision.transforms as transforms
import torchvision.models as models

# Standard Libraries
from PIL import Image
import json
import requests

# Third-Party Libraries
import numpy as np
import matplotlib.pyplot as plt
import os

# CrypTen
import crypten
import crypten.mpc as mpc

# Initializing CrypTen
crypten.init()
torch.set_num_threads(1)

In [ ]:
def process_image(output_dir):
    """ 
    Takes the original picture and prepares it for ResNet18
    Matches the size to 224x 224, transforms it to a tensor and normalizes it with recommended means and std for every colour channel
    Creates the noisy pictures needed for robustness
    Saves the pictures in the given file 
    Pictures are either the processed or noisy one
    """
    # Actual transformation
    for i in range(10):
        transform_pic = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
        image_path=f"test_{i}_image.jpg"
        with Image.open(image_path) as img:
            img_processed = transform_pic(img.convert("RGB"))#.unsqueeze(0) 

        # Inverse Transformation (just for testing)
        inv_transform = transforms.Compose([
            transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                               std=[1/0.229, 1/0.224, 1/0.225]),
            transforms.ToPILImage()
        ])
    
        # Save picture
        torch.save(img_processed, os.path.join(output_dir, f'processed_{i}.pth'))

        img_path = os.path.join(output_dir, f'processed_{i}.pth')
        if not os.path.exists(img_path):
            continue
            
        image = torch.load(img_path)
        noise = torch.randn_like(image) * 0.1
        noisy_image = image + noise
        torch.save(noisy_image, os.path.join(output_dir, f'noisy_pic_{i}.pth'))
        plt.imshow(inv_transform(noisy_image))
        plt.show
    
process_image('noisy_images')

In [ ]:
# Helper functions for the code
def calculate_gini_index(heatmap):
    """Calculate Gini index for sparsity """
    flattened = heatmap.flatten().abs()
    flattened += 1e-10  # to avoid division by zero
    flattened = torch.sort(flattened).values
    n = flattened.shape[0]
    index = torch.arange(1, n+1)
    return 1 - 2 * (torch.sum((flattened * (n - index + 0.5))) / (n * torch.sum(flattened)))

def calculate_sens_value(heatmap_clean, heatmap_noisy):
    """Calculate normalized SENS value"""
    diff = heatmap_clean - heatmap_noisy
    l2_diff = torch.linalg.norm(diff)
    l2_clean = torch.linalg.norm(heatmap_clean)
    return l2_diff / (l2_clean + 1e-10)  # to avoid division by zero

## The sliding window algorithm

This function implements the unencrypted version of the sliding-window algorithm. 

First it loads the model and the picture that should be classified by the model. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset. 

Next, the parameters of the occlusion analysis are defined. The patch_size is the size of the patch that will be used to mask the regions of the picture. The patch_size doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. These two parameters can be changed, especially the size of the patch. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps_dec"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def occlusion_analysis():
    """
    Performs the standard case of occlusion analysis on ResNet18.
    The algorithm is a version of the sliding-window algorithm.
    The result is a heatmap.
    """
    # Load model
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # Pre-trained version of ResNet18
    model.eval()  # Put model in inference phase

    results = []
    
    for i in range(num_images):
    
        # Load data
        image = torch.load(f'noisy_images/processed_{i}.pth').unsqueeze(0)

        # Load noisy image
        noisy_img = torch.load(f'noisy_images/noisy_pic_{i}.pth').unsqueeze(0)
    
        # Parameters for Occlusion Analysis 
        # - patch size
        # - occluded_value
        # - heatmap
        patch_size = 112 # Can be changed
        occluded_value = 0 # Can be changed

        heatmaps = {}
        for mode, current_data in [('clean', image), ('noisy', noisy_img)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = model(current_data)

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_pred = original_output.argmax().item() #index of the class with highest probability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap = torch.zeros((current_data.shape[2], current_data.shape[3]))
        
            num_it = 1 # Counter for number of iterations
        
            # Iterating over the picture
            for y in range(0, current_data.shape[2], patch_size):
                for x in range(0, current_data.shape[3], patch_size):
                    # Edge treatment:
                    # The normal patch size is used 
                    # Unless the patch doesn't entirely fit in the edge
                    # Then the patch_size is the image.size - the current pixel position
                    better_patch_size_y = min(patch_size, current_data.shape[2] - y)
                    better_patch_size_x = min(patch_size, current_data.shape[3] - x)
                    
                    # Used if patch would be too small
                    if better_patch_size_y < 1 or better_patch_size_x < 1:
                        continue
                        
                    # Clone the picture and put the mask on the position
                    occluded_image = current_data.clone()
                    occluded_image[:, :, y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
        
                    # Calculating the occluded_output
                    with torch.no_grad():
                        occluded_output = model(occluded_image)
                    
                    # Calculating the absolute difference of the original output and the occluded output
                    diff = (original_output - occluded_output).abs().sum().item()
        
                    # Put the difference in the heatmap
                    heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff
        
                    crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                    num_it += 1
        
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap and save it
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()

occlusion_analysis()

## The hierarchical occlusion algorithm

This function implements the unencrypted version of the hierarchical occlusion algorithm. 

First it loads the model and the picture that should be classified by the model. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset.    

Next, the parameters of the occlusion analysis are defined. The initial_patch is the size of the patch that will be used to mask the regions of the picture and is the starting size of the patch. The initial_patch doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The initial_patch should be dividable by 2 to avoid artifacts in the heatmap. The minimal_patch is the size of the smallest patch, so it should be smaller than the initial patch. However, the minimal_patch cannot always be the minimal patch; e.g., if the patch_size is 6 and the minimal patch is 2, the smallest patch will be 3 to avoid artifacts within the heatmap. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. The threshold is needed to make the decision if a patch should be divided or not. It can be any number between 0 and 1. These four parameters can be changed, especially the size of the patch, the minimal patch and the threshold. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps_dec"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def hierarchical_occlusion():
    """
    Performs a hierarchical occlusion analysis on ResNet18.
    The algorithm identifies regions that are important for the model and partitions the patch.
    The result is a heatmap.
    """
    # Load model
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # Pre-trained version of ResNet18
    model.eval()  # Put model in inference phase

    results = []
    
    for i in range(num_images):
    
        # Load data
        image = torch.load(f'noisy_images/processed_{i}.pth').unsqueeze(0)

        # Load noisy image
        noisy_img = torch.load(f'noisy_images/noisy_pic_{i}.pth').unsqueeze(0)
    
        # Parameters for Occlusion Analysis 
        # - initial_patch
        # - minimal_patch
        # - occluded_value
        # - threshold
        # - heatmap
        initial_patch = 112 # Can be changed
        minimal_patch = 56 # Can be changed
        occluded_value = 0 # Can be changed
        threshold = 0.5 # Can be changed

        heatmaps = {}
        for mode, current_data in [('clean', image), ('noisy', noisy_img)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = model(current_data)#.view(1, 784)) # View is needed for the input of the model

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_pred = original_output.argmax().item() #index of the class with highest probability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap = torch.zeros((current_data.shape[2], current_data.shape[3]))
            
            num_it = 1 # Counter for number of iterations
            
            def patch_partition(starting_point, current_patch_size, heatmap):
                """
                Partitions the patch if the difference of the original output and occluded output is important.
                Args:
                    starting_point: top-left corner of the current patch
                    current_patch_size: Size of the current patch
                    heatmap: Contains the difference of the original output and occluded output
                Returns:
                    heatmap
                """  
                nonlocal num_it
                y, x = starting_point
        
                # Edge treatment:
                # The normal patch size is used 
                # Unless the patch doesn't entirely fit in the edge
                # Then the patch_size is the image.size - the current pixel position
                better_patch_size_y = min(current_patch_size, current_data.shape[2] - y)
                better_patch_size_x = min(current_patch_size, current_data.shape[3] - x)
        
                # Used if patch would be too small
                if better_patch_size_y < 1 or better_patch_size_x < 1:
                    return heatmap      
        
                # Clone the picture and put the mask on the position
                occluded_image = current_data.clone()
                occluded_image[:, :, y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
        
                # Calculating the occluded_output
                with torch.no_grad():
                    occluded_output = model(occluded_image)
                    
                # Calculating the absolute difference of the original output and the occluded output
                diff = (original_output - occluded_output).abs().sum().item()
        
                # Put the difference in the heatmap
                heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff
        
                # Calculates the difference but in the range 0 to 1
                comp = ((original_output.softmax(1) - occluded_output.softmax(1)).abs().sum())/ 2
        
                # Decision if the patch should be divided
                # Difference needs to be bigger than threshold 
                # and patch must be bigger than smallest patch
                differentiate = (comp > threshold) and (float(current_patch_size)/2.0 >= float(minimal_patch))
                
                # If decision is true, divide the patch 
                # Set the position for the patches
                if differentiate:
                    half_patch = current_patch_size // 2
                    # Calculate possible positions and set new positions
                    for poss_y in [0, half_patch]:
                        for poss_x in [0, half_patch]:
                            new_y = poss_y + y
                            new_x = poss_x + x
                            # Check if patch position is bigger than picture
                            if (new_y < current_data.shape[2] and new_x < current_data.shape[3]):
                                heatmap = patch_partition((new_y, new_x), half_patch, heatmap)
                
                crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                num_it += 1
                
                return heatmap
            
            # Start of Occlusion Analysis
            for y in range(0, current_data.shape[2], initial_patch):
                for x in range(0, current_data.shape[3], initial_patch):
                    heatmap = patch_partition((y,x), initial_patch, heatmap)          
                    
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()

hierarchical_occlusion()

## The Monte Carlo Occlusion Algorithm

This function implements the unencrypted version of the Monte Carlo algorithm. 

First it loads the model and the picture that should be classified by the model. All of it is done without encryption. Since the model is trained on the ImageNet dataset, the picture needs to be of size 224x224, which is provided by the function in the preset. 

Next, the parameters of the occlusion analysis are defined. The patch_size is the size of the patch that will be used to mask the regions of the picture. The patch_size doesn't need to fit in the picture entirely since there is edge treatment, but it needs to be smaller than the picture. The occluded_value is the value of the mask used for the occlusion. In the standard case, it is set to black. The num_simulations is the number of simulations that the algorithm will do. This number shouldn't be set too small to achieve good results. Note that it isn't guaranteed that all the regions in the heatmap will be covered. These three parameters can be changed, especially the size of the patch and the number of iterations. The last parameter is the heatmap. It is the same size as the picture, and at the beginning it is black.

The code loops over every picture and computes the heatmap for the not-noisy picture and the noisy picture. Both of the heatmaps get saved and the Gini-Index of the clean heatmap and the sens between the clean and noisy heatmap get calculated. In the end, the results will be printed and statistics will be visualized. 

In [ ]:
save_dir = "heatmaps_dec"
clean_images_dir = "/tmp/clean_images"
noisy_images_dir = "/tmp/noisy_images"
num_images = 10 # Number needs to be adjusted according to number of pictures

# Create directories
os.makedirs(save_dir, exist_ok=True)

# Worldsize is 2 because of the two parties
@mpc.run_multiprocess(world_size=2)
def monte_carlo_occlusion_analysis():
    """
    Performs a Monte Carlo occlusion analysis on ResNet18.
    The algorithm sets the patch to random places on the picture and calculates the difference.
    The result is a statistical heatmap.
    """
    # Set seeds for results  
    torch.manual_seed(42)
    np.random.seed(42)

    results=[]
    
    # Load model
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) # Pre-trained version of ResNet18
    model.eval()  # Put model in inference phase

    for i in range(num_images):
    
        # Load data
        image = torch.load(f'noisy_images/processed_{i}.pth').unsqueeze(0)

        # Load noisy image
        noisy_img = torch.load(f'noisy_images/noisy_pic_{i}.pth').unsqueeze(0)

        # Parameters for Occlusion Analysis 
        # - patch_size
        # - occluded_value
        # - num_simulations
        # - encrypted heatmap
        patch_size = 100 # Can be changed
        occluded_value = 0 # Can be changed
        num_simulations = 7 # Can be changed
        
        heatmaps = {}
        for mode, current_data in [('clean', image), ('noisy', noisy_img)]:
    
            # Computation of original output of the original picture
            with torch.no_grad(): # Allows quicker computation
                original_output = model(current_data)#.view(1, 784)) # View is needed for the input of the model

            url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
            class_idx = requests.get(url).json()
            def get_class_name(class_id):
                return class_idx[str(class_id)][1] 
        
            original_pred = original_output.argmax().item() #index of the class with highest probability
            predicted_class_name = get_class_name(original_pred)       
            crypten.print(f'Image {i} {mode} - Actual prediction: {predicted_class_name}')
            
            heatmap = torch.zeros((current_data.shape[2], current_data.shape[3]))
    
            num_it = 1 # Counter for number of iterations
        
            # Actual_sim represents the number of actual simulations
            # Tried_sim is the number of tried simulations (those who failed and those who didn't
            # Max_sim is the number of maximum tries to avoid endless loop
            actual_sim = 0
            tried_sim = 0
            max_sim = num_simulations * 2 # Can be changed
        
            # Calculate the possible positions for the patch
            # By dividing the possible positions by the patch_size
            num_patches_y = (current_data.shape[2] - patch_size) // patch_size + 1
            num_patches_x = (current_data.shape[3] - patch_size) // patch_size + 1
        
            while actual_sim < num_simulations and tried_sim < max_sim:
                # Choose a position for the patch
                random_patch_y = np.random.randint(0, num_patches_y + 1)
                random_patch_x = np.random.randint(0, num_patches_x + 1)
        
                # Calculate the positions with the patch_size
                y = random_patch_y * patch_size
                x = random_patch_x * patch_size
        
                # Edge treatment:
                # The normal patch size is used 
                # Unless the patch doesn't entirely fit in the edge
                # Then the patch_size is the image.size - the current pixel position
                better_patch_size_y = min(patch_size, current_data.shape[2] - y)
                better_patch_size_x = min(patch_size, current_data.shape[3] - x)
        
                # Used if patch would be too small
                if better_patch_size_y < 1 or better_patch_size_x < 1:
                        tried_sim += 1
                        continue
                    
                # Clone the picture and put the mask on the position
                occluded_image = current_data.clone()
                occluded_image[:, :, y : y + better_patch_size_y, x : x + better_patch_size_x] = occluded_value
        
                # Calculating the occluded_output
                with torch.no_grad():
                    occluded_output = model(occluded_image)
                    
                # Calculating the absolute difference of the original output and the occluded output
                diff = (original_output - occluded_output).abs().sum().item()
        
                # Put the difference in the heatmap
                heatmap[y : y + better_patch_size_y, x : x + better_patch_size_x] += diff
        
                crypten.print(f"\rNumber of iterations: {num_it}", end='', flush=True)
                num_it += 1
                tried_sim += 1
                actual_sim += 1
        
            # Calculate the average influence
            heatmap /= actual_sim
        
            # Normalize heatmap
            heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
            heatmaps[mode] = heatmap_norm
            
            # Save heatmap
            torch.save(heatmap_norm, f'{save_dir}/heatmap_{mode}_{i}.pt')
            
            # Calculate metrics
            if mode == 'clean':
                gini = calculate_gini_index(heatmap_norm)
            results.append({
                'image_id': i,
                'mode': mode,
                'heatmap': heatmap_norm,
                'gini_index': gini.item(),
                'prediction': predicted_class_name
            })
                
            # Visualize heatmap
            plt.figure(figsize=(10, 5))
            plt.imshow(heatmap_norm, cmap='hot')
            plt.title(f"Heatmap {mode} Image {i} (Pred: {predicted_class_name})")
            plt.colorbar()
            plt.savefig(f'{save_dir}/heatmap_{mode}_{i}.png')
            plt.close()
            
        # Calculate SENS value between clean and noisy heatmaps
        sens = calculate_sens_value(heatmaps['clean'], heatmaps['noisy'])
        
        # Update results for clean image
        for res in results:
            if res['image_id'] == i and res['mode'] == 'clean':
                res['sens'] = sens.item()
                break
    
    # Print summary statistics
    clean_ginis = [res['gini_index'] for res in results if res['mode'] == 'clean']
    sens_values = [res['sens'] for res in results if 'sens' in res and res['mode'] == 'clean']
    id_values = [res['image_id'] for res in results if res['mode'] == 'clean']
    
    crypten.print("\nSummary Statistics:")
    crypten.print(f"Average Gini index (clean): {np.mean(clean_ginis):.4f} ± {np.std(clean_ginis):.4f}, Max:{np.max(clean_ginis):.4f}, Min:{np.min(clean_ginis):.4f}")
    crypten.print(f"Average SENS value: {np.mean(sens_values):.4f} ± {np.std(sens_values):.4f}, Max:{np.max(sens_values):.4f}, Min:{np.min(sens_values):.4f}")

    plt.figure(figsize=(10, 5))
    plt.plot(clean_ginis, 'o-')
    plt.show()
    
    plt.figure(figsize=(10, 5))
    plt.plot(sens_values, 'o-')
    plt.show()

monte_carlo_occlusion_analysis()

This code is used for cleaning the files up.

In [ ]:
import os

filenames = ['/tmp/processed_image_dec.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)